# Embedding Evaluation — Metadata Notebook

End-to-end walkthrough using `emb_tight_meta.json` and `emb_sparse_meta.json`. Both files supply a `metadata` field alongside embeddings, enabling graded nDCG and per-attribute KPIs.

**Tight** — orthogonal class prototypes, tiny per-sample noise. Intra-class cosine ≈ 0.99, inter-class ≈ 0.00. All KPIs near-perfect.

**Sparse** — nearby class prototypes (~60° apart), large per-sample noise. Classes overlap heavily — gap ≈ 0.07, purity@5 ≈ 0.56.

Each image belongs to a metadata group (`corn_HB-25000SBC`, `corn_nikon_d610`, etc.) that carries `class_name` and `attributes` (`camera`, `growth_stage`). This unlocks:

* `knn_metadata_ndcg` — graded 0–3 relevance (explicit positive > same class + attrs > same class > other)
* `knn_attribute_ndcg` — one binary nDCG per attribute key (`camera`, `growth_stage`)

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from pai.ag_emb.schemas.evaluate import MetadataGroup
from pai.ag_emb.services.evaluate import run_evaluation
from pai.ag_emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_lle,
    plot_tsne,
    print_result,
)

---

## Tight Scenario

### Load Data

In [ ]:
payload_path = Path("emb_tight_meta.json")
if not payload_path.exists():
    raise FileNotFoundError("emb_tight_meta.json not found — start Jupyter from the examples/ directory")

with open(payload_path) as f:
    payload = json.load(f)

embeddings: dict[str, list[float]] = payload["embeddings"]
metadata: dict[str, MetadataGroup] = {key: MetadataGroup(**group) for key, group in payload["metadata"].items()}

print(f"Loaded {len(embeddings)} embeddings  (dim={len(next(iter(embeddings.values())))})")
print(f"Metadata groups: {list(metadata.keys())}")

### Run Evaluation

In [ ]:
result = run_evaluation(
    image_embeddings=embeddings,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
    metadata=metadata,
)

print_result(result)

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

#### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class. The diagonal equals mean KNN purity — higher is better.

In [ ]:
plot_knn_confusion(result, output_path=None)

#### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class. Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [ ]:
plot_cosine_similarity(embeddings, result, output_path=None)

#### t-SNE — 2D

t-SNE preserves local neighborhood structure. Well-separated clusters indicate the model has learned class-discriminative features.

In [ ]:
plot_tsne(embeddings, result, output_path=None, dimensions=2)

#### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [ ]:
plot_tsne(embeddings, result, output_path=None, dimensions=3)

#### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view. Drag to rotate.

In [ ]:
plot_lle(embeddings, result, output_path=None)

---

## Sparse Scenario

Same image paths, metadata groups, and attribute structure as the tight scenario, but class prototypes are close together (~60° apart) and per-sample noise is large. Compare KPIs and plots directly against the tight scenario above to see how embedding quality degrades.

### Load Data

In [ ]:
sparse_path = Path("emb_sparse_meta.json")
with open(sparse_path) as f:
    sparse_payload = json.load(f)

sparse_embeddings: dict[str, list[float]] = sparse_payload["embeddings"]
sparse_metadata: dict[str, MetadataGroup] = {
    key: MetadataGroup(**group) for key, group in sparse_payload["metadata"].items()
}

print(f"Loaded {len(sparse_embeddings)} sparse embeddings  (dim={len(next(iter(sparse_embeddings.values())))})")
print(f"Metadata groups: {list(sparse_metadata.keys())}")

### Run Evaluation

In [ ]:
sparse_result = run_evaluation(
    image_embeddings=sparse_embeddings,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
    metadata=sparse_metadata,
)

print_result(sparse_result)

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

#### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class. The diagonal equals mean KNN purity — higher is better.

In [ ]:
plot_knn_confusion(sparse_result, output_path=None)

#### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class. Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [ ]:
plot_cosine_similarity(sparse_embeddings, sparse_result, output_path=None)

#### t-SNE — 2D

t-SNE preserves local neighborhood structure. Well-separated clusters indicate the model has learned class-discriminative features.

In [ ]:
plot_tsne(sparse_embeddings, sparse_result, output_path=None, dimensions=2)

#### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [ ]:
plot_tsne(sparse_embeddings, sparse_result, output_path=None, dimensions=3)

#### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view. Drag to rotate.

In [ ]:
plot_lle(sparse_embeddings, sparse_result, output_path=None)